# NYC Mobility - Source Ingestion

## What ingestion means

Ingestion is how we bring source data into storage before transforming it. This notebook uses repeatable source downloads, deterministic filenames, content hashes, and metadata sidecars. Existing raw files are preserved; do not delete them just to create a clean-looking rerun.

## Shared idempotent landing helper

Run this cell first. Raw landing uses a first-write-wins rule. If the deterministic file already exists, the helper validates its SHA-256 hash against the metadata sidecar and returns `IDEMPOTENT_SKIP` without downloading or overwriting anything. A missing file is downloaded once and recorded with its metadata. To capture a deliberate source revision, use a new versioned filename instead of replacing preserved raw data.

In [0]:
%python
import hashlib
import json
import requests
from datetime import datetime, timezone
from pathlib import Path

REQUEST_HEADERS = {
    "User-Agent": "FTW-B12-Data-Engineering-Course-Project"
}

def fetch_and_land(source_url, target_path, source_system, timeout=120):
    target_path = Path(target_path)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    metadata_path = Path(f"{target_path}.metadata.json")

    # First-write-wins idempotency: a rerun reuses the preserved raw file.
    if target_path.exists():
        raw_content = target_path.read_bytes()
        if not raw_content:
            raise ValueError(f"Existing raw file is empty: {target_path}")

        content_hash = hashlib.sha256(raw_content).hexdigest()
        if metadata_path.exists():
            metadata = json.loads(metadata_path.read_text())
            if metadata.get("sha256") != content_hash:
                raise RuntimeError(
                    f"Metadata hash mismatch for {metadata_path.name}; "
                    "review the preserved raw file and sidecar."
                )
        else:
            metadata = {
                "source_system": source_system,
                "source_url": source_url,
                "source_file": target_path.name,
                "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
                "http_status": None,
                "content_type": None,
                "bytes_received": len(raw_content),
                "sha256": content_hash,
                "metadata_rebuilt_from_existing_file": True,
            }
            metadata_path.write_text(json.dumps(metadata, indent=2))

        evidence = {
            "action": "IDEMPOTENT_SKIP",
            "http_status": metadata.get("http_status"),
            "source_file": target_path.name,
            "output_path": str(target_path),
            "metadata_path": str(metadata_path),
            "bytes_received": len(raw_content),
            "sha256": content_hash,
        }
        print(json.dumps(evidence, indent=2))
        return evidence

    response = requests.get(
        source_url,
        timeout=timeout,
        headers=REQUEST_HEADERS,
    )
    response.raise_for_status()
    raw_content = response.content
    if not raw_content:
        raise ValueError(f"Empty response received from {source_url}")

    content_hash = hashlib.sha256(raw_content).hexdigest()
    target_path.write_bytes(raw_content)

    metadata = {
        "source_system": source_system,
        "source_url": source_url,
        "source_file": target_path.name,
        "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
        "http_status": response.status_code,
        "content_type": response.headers.get("Content-Type"),
        "bytes_received": len(raw_content),
        "sha256": content_hash,
    }
    metadata_path.write_text(json.dumps(metadata, indent=2))

    evidence = {
        "action": "WRITTEN",
        "http_status": response.status_code,
        "source_file": target_path.name,
        "output_path": str(target_path),
        "metadata_path": str(metadata_path),
        "bytes_received": len(raw_content),
        "sha256": content_hash,
    }
    print(json.dumps(evidence, indent=2))
    return evidence


## Green Taxi

NYC TLC publishes Green Taxi trip records as monthly Parquet files. Select March, April, or May with the widget, then land only that month using the official deterministic filename. This replaces the undocumented manual-only landing step.

In [0]:
%python
allowed_taxi_months = ["2026-03", "2026-04", "2026-05"]
try:
    green_taxi_month = dbutils.widgets.get("green_taxi_month").strip()
except Exception:
    dbutils.widgets.dropdown(
        "green_taxi_month",
        "2026-03",
        allowed_taxi_months,
        "Green Taxi month",
    )
    green_taxi_month = dbutils.widgets.get("green_taxi_month").strip()

if green_taxi_month not in allowed_taxi_months:
    raise ValueError(f"green_taxi_month must be one of {allowed_taxi_months}")

green_taxi_filename = f"green_tripdata_{green_taxi_month}.parquet"
green_taxi_url = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    f"{green_taxi_filename}"
)
green_taxi_path = (
    Path("/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/")
    / "groups/week-08/group-b/source/green_taxi"
    / green_taxi_filename
)

green_taxi_evidence = fetch_and_land(
    green_taxi_url,
    green_taxi_path,
    source_system="nyc_tlc",
)


The Green Taxi baseline returned HTTP 200 for `green_tripdata_2026-03.parquet` and reported `IDEMPOTENT_SKIP`, confirming that its content matched the preserved file. The recorded metadata includes the deterministic path, 1,082,530-byte response size, and SHA-256 hash.

March established the baseline, while April and May were added successfully as incremental monthly Bronze batches. Together, the three approved Green Taxi files provide complete March–May ingestion coverage.

## Taxi Zones

The Taxi Zone lookup is downloaded from the official NYC TLC link and saved with the exact filename `taxi_zone_lookup.csv`. The deterministic path and hash check make the landing step repeatable.


In [0]:
%python
taxi_zone_url = (
    "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
)
taxi_zone_path = (
    Path("/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/")
    / "groups/week-08/group-b/source/taxi_zones"
    / "taxi_zone_lookup.csv"
)

taxi_zone_evidence = fetch_and_land(
    taxi_zone_url,
    taxi_zone_path,
    source_system="nyc_tlc",
)


The Taxi Zone cell returned HTTP 200 for `taxi_zone_lookup.csv` and reported `IDEMPOTENT_SKIP`, confirming that the downloaded content matched the preserved file. The saved output includes the metadata path, 12,331-byte response size, and SHA-256 hash.

## Weather API ingestion

We use the Open-Meteo Archive API because the assignment period is historical: March through May 2026. The current Bronze source is the existing combined file `open_meteo_2026-03-01_2026-05-31.json`, which contains the full March-May period and remains one raw JSON record for that saved batch in Bronze. The parameterized cell below can also land individual months when a future monthly ingestion run is needed. Raw responses are saved without flattening because hourly expansion, cleaning, and reshaping belong in Silver.

In [0]:
%python
import calendar
from datetime import datetime

allowed_months = ["2026-03", "2026-04", "2026-05"]
try:
    weather_month = dbutils.widgets.get("weather_month").strip()
except Exception:
    dbutils.widgets.dropdown("weather_month", "2026-03", allowed_months, "Weather month")
    weather_month = dbutils.widgets.get("weather_month").strip()

if weather_month not in allowed_months:
    raise ValueError(f"weather_month must be one of {allowed_months}")

month_start = datetime.strptime(weather_month, "%Y-%m").date()
month_end_day = calendar.monthrange(month_start.year, month_start.month)[1]
start_date = month_start.isoformat()
end_date = month_start.replace(day=month_end_day).isoformat()

weather_url = (
    f"https://archive-api.open-meteo.com/v1/archive"
    f"?latitude=40.7128"
    f"&longitude=-74.006"
    f"&start_date={start_date}"
    f"&end_date={end_date}"
    f"&hourly=temperature_2m,precipitation,rain,snowfall,weather_code,wind_speed_10m"
    f"&timezone=America%2FNew_York"
)

weather_filename = f"open_meteo_{start_date}_{end_date}.json"
weather_path = (
    Path("/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/")
    / "groups/week-08/group-b/source/weather"
    / weather_filename
)

print("Selected month:", weather_month)
print("Date range:", start_date, "to", end_date)
weather_evidence = fetch_and_land(
    weather_url,
    weather_path,
    source_system="open_meteo",
)


The Weather cell returned HTTP 200 for March 2026 and reported `WRITTEN`. It saved `open_meteo_2026-03-01_2026-03-31.json` and its metadata sidecar, including the recorded 34,007-byte response size and SHA-256 hash.

The approved Bronze Weather source remains `open_meteo_2026-03-01_2026-05-31.json`, which contains the complete March–May period as one raw JSON batch. The same-file Bronze rerun confirms idempotent handling of this selected combined source.


## Ingestion Completion Summary

Green Taxi and Taxi Zones use repeatable downloads from the official NYC TLC links. Green Taxi and Weather are parameterized by month. Deterministic filenames, metadata sidecars, and SHA-256 checks preserve the raw sources and identify unchanged content through `IDEMPOTENT_SKIP`.

The completed source coverage is:

1. Green Taxi March established the baseline; April and May were added as incremental monthly Bronze batches.
2. Taxi Zones uses the exact source file `taxi_zone_lookup.csv`.
3. Weather uses `open_meteo_2026-03-01_2026-05-31.json` as the single approved raw Bronze batch covering March–May.
4. Same-file Bronze reruns confirm that unchanged inputs do not duplicate target records.

The additional March-only Weather file demonstrates the parameterized ingestion path but is excluded from the approved Bronze baseline. Raw files and metadata sidecars remain preserved as lineage evidence, while cleaning and hourly Weather expansion remain in Silver.
